<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/03_breakhis_formal_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Formal Development-Only Model Selection

## CIS 631 — Breast Histopathology Calibration Research

This notebook implements the formal model-development stage of our breast histopathology calibration study.

The objective is to compare fine-tuned ResNet-18 and EfficientNet-B0 using the previously established patient-grouped three-fold Development partition.

### Experimental restrictions

- Only the **49 BreakHis Development patients** can be used for CNN model development and model selection.
- Only BreakHis **40× images** are used for primary model development.
- Existing patient assignments, Development folds, preprocessing parameters, and patch coordinates must remain unchanged.
- Calibration patients are reserved for fitting post-hoc calibrators after final CNN training.
- Test patients and external BACH/BRACS datasets are excluded from model development.
- Model selection will be based on Development-fold validation results.

The preprocessing parameters remain fixed:

| Parameter | Value |
|---|---|
| Source physical resolution | 0.49 µm/pixel |
| Target physical resolution | 0.50 µm/pixel |
| Patch size | 224 × 224 |
| Grid | 4 × 4 |
| Maximum patches per image | 16 |
| Optical density threshold | 0.08 |
| Saturation threshold | 0.05 |
| Minimum tissue fraction | 0.20 |

### Stage A: Data and fold verification

Before training, we reconstruct the complete Development patch manifest from previously saved patch manifests and verify that all three folds preserve patient-level separation.

No CNN training or held-out performance evaluation is performed during Stage A.


In [ ]:

# ============================================================
# Stage A — Environment and repository setup
# ============================================================

from pathlib import Path
import subprocess
import random

import numpy as np
import pandas as pd

from IPython.display import display

SEED = 631

random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

REPO_URL = (
    "https://github.com/Yisihaq01/"
    "breast-histology-calibration.git"
)

if not REPO_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_ROOT)
        ],
        check=True
    )
else:
    print(
        "Repository folder already exists."
    )

METADATA_DIR = REPO_ROOT / "Metadata"

RESULTS_DIR = (
    REPO_ROOT /
    "results" /
    "formal_development"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Repository:", REPO_ROOT)
print("Seed:", SEED)
print("Metadata directory:", METADATA_DIR)
print("Results directory:", RESULTS_DIR)


Repository: /content/breast-histology-calibration
Seed: 631
Metadata directory: /content/breast-histology-calibration/Metadata
Results directory: /content/breast-histology-calibration/results/formal_development


In [ ]:
# ============================================================
# Diagnose missing repository metadata
# ============================================================

from pathlib import Path
import subprocess

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

def git_check(*args):
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        capture_output=True,
        text=True
    )
    print(result.stdout or result.stderr)

print("1. Current branch:")
git_check("branch", "--show-current")

print("\n2. Available remote branches:")
git_check("branch", "-r")

print("\n3. Latest commit:")
git_check("log", "-1", "--oneline")

print("\n4. Tracked metadata files:")
git_check("ls-files", "metadata")

print("\n5. Local metadata directory:")

metadata_dir = REPO_ROOT / "metadata"

if metadata_dir.exists():
    for path in sorted(metadata_dir.iterdir()):
        print(path.name)
else:
    print("Metadata directory does not exist.")

print("\n6. Repository status:")
git_check("status", "--short", "--branch")

1. Current branch:
main


2. Available remote branches:
  origin/Calibration_analysis
  origin/HEAD -> origin/main
  origin/main


3. Latest commit:
ae6f0d2 Merge branch 'main' of https://github.com/Yisihaq01/breast-histology-calibration


4. Tracked metadata files:


5. Local metadata directory:
Metadata directory does not exist.

6. Repository status:
## main...origin/main



In [ ]:
# ============================================================
# Check which GitHub branches contain the frozen metadata
# ============================================================

import subprocess
from pathlib import Path

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

required_names = [
    "breakhis_40x_final_metadata.csv",
    "breakhis_patient_split.csv",
    "breakhis_development_folds.csv",
    "baseline_train_patch_manifest.csv",
    "baseline_val_patch_manifest.csv",
]

branches = [
    "origin/main",
    "origin/Calibration_analysis"
]

for branch in branches:

    result = subprocess.run(
        [
            "git",
            "-C", str(REPO_ROOT),
            "ls-tree",
            "-r",
            "--name-only",
            branch
        ],
        capture_output=True,
        text=True,
        check=True
    )

    tracked_files = result.stdout.splitlines()

    print(f"\n===== {branch} =====")

    for filename in required_names:

        matches = [
            path for path in tracked_files
            if Path(path).name == filename
        ]

        if matches:
            print("FOUND:", matches[0])
        else:
            print("MISSING:", filename)

# Check if Git is ignoring the metadata directory
print("\n===== Git ignore check =====")

result = subprocess.run(
    [
        "git",
        "-C", str(REPO_ROOT),
        "check-ignore",
        "-v",
        "--no-index",
        "metadata/breakhis_patient_split.csv"
    ],
    capture_output=True,
    text=True
)

print(
    result.stdout.strip()
    or "No matching ignore rule found."
)


===== origin/main =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== origin/Calibration_analysis =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== Git ignore check =====
No matching ignore rule found.


In [ ]:
# ============================================================
# Verify corrected GitHub metadata directory
# ============================================================

from pathlib import Path

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

METADATA_DIR = REPO_ROOT / "Metadata"

print("Metadata directory:", METADATA_DIR)
print("Directory exists:", METADATA_DIR.exists())

required_files = {
    "final_40x_metadata":
        METADATA_DIR / "breakhis_40x_final_metadata.csv",

    "patient_split":
        METADATA_DIR / "breakhis_patient_split.csv",

    "development_folds":
        METADATA_DIR / "breakhis_development_folds.csv",

    "training_patch_manifest":
        METADATA_DIR / "baseline_train_patch_manifest.csv",

    "validation_patch_manifest":
        METADATA_DIR / "baseline_val_patch_manifest.csv"
}

for name, filepath in required_files.items():
    print(
        f"{name}:",
        "FOUND" if filepath.exists() else "MISSING"
    )

assert all(
    path.exists()
    for path in required_files.values()
)

print(
    "\nPASS: All five frozen metadata files located."
)

Metadata directory: /content/breast-histology-calibration/Metadata
Directory exists: True
final_40x_metadata: FOUND
patient_split: FOUND
development_folds: FOUND
training_patch_manifest: FOUND
validation_patch_manifest: FOUND

PASS: All five frozen metadata files located.


In [ ]:

# ============================================================
# Stage A — Inspect frozen metadata structure
# ============================================================

import pandas as pd

for name, filepath in required_files.items():

    df = pd.read_csv(filepath)

    print("\n" + "=" * 55)
    print(name.upper())
    print("=" * 55)

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if name == "development_folds":
        print("\nFirst 5 fold assignments:")
        display(df.head(5))

    elif name == "patient_split":
        print("\nPatient split counts:")
        print(df["split"].value_counts())

    elif name in [
        "training_patch_manifest",
        "validation_patch_manifest"
    ]:
        print("\nFirst 2 patch records:")
        display(df.head(2))



FINAL_40X_METADATA
Rows: 1961
Columns: ['filename', 'relative_path', 'patient_id', 'diagnosis_code', 'diagnosis', 'original_subtype_code', 'original_subtype', 'subtype_code', 'subtype', 'magnification', 'image_number', 'split']

PATIENT_SPLIT
Rows: 81
Columns: ['patient_id', 'diagnosis_code', 'diagnosis', 'subtype_code', 'subtype', 'split']

Patient split counts:
split
development    49
calibration    16
test           16
Name: count, dtype: int64

DEVELOPMENT_FOLDS
Rows: 49
Columns: ['patient_id', 'diagnosis_code', 'diagnosis', 'subtype_code', 'subtype', 'split', 'development_fold']

First 5 fold assignments:


,patient_id,diagnosis_code,diagnosis,subtype_code,subtype,split,development_fold
0,14-29960CD,B,Benign,A,Adenosis,development,1
1,14-14134,B,Benign,F,Fibroadenoma,development,1
2,14-21998CD,B,Benign,F,Fibroadenoma,development,1
3,14-13200,B,Benign,TA,Tubular adenoma,development,1
4,14-3411F,B,Benign,TA,Tubular adenoma,development,1



TRAINING_PATCH_MANIFEST
Rows: 12034
Columns: ['patient_id', 'filename', 'relative_path', 'diagnosis', 'label', 'patch_index', 'x', 'y', 'tissue_fraction', 'used_fallback']

First 2 patch records:


,patient_id,filename,relative_path,diagnosis,label,patch_index,x,y,tissue_fraction,used_fallback
0,14-10147,SOB_M_MC-14-10147-40-009.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Malignant,1,0,0,0,0.578823,False
1,14-10147,SOB_M_MC-14-10147-40-009.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Malignant,1,1,154,0,0.518634,False



VALIDATION_PATCH_MANIFEST
Rows: 5861
Columns: ['patient_id', 'filename', 'relative_path', 'diagnosis', 'label', 'patch_index', 'x', 'y', 'tissue_fraction', 'used_fallback']

First 2 patch records:


,patient_id,filename,relative_path,diagnosis,label,patch_index,x,y,tissue_fraction,used_fallback
0,14-13200,SOB_B_TA-14-13200-40-012.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Benign,0,0,0,0,0.928133,False
1,14-13200,SOB_B_TA-14-13200-40-012.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Benign,0,1,154,0,0.830716,False


In [ ]:
# ============================================================
# Check which GitHub branches contain the frozen metadata
# ============================================================

import subprocess
from pathlib import Path

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

required_names = [
    "breakhis_40x_final_metadata.csv",
    "breakhis_patient_split.csv",
    "breakhis_development_folds.csv",
    "baseline_train_patch_manifest.csv",
    "baseline_val_patch_manifest.csv",
]

branches = [
    "origin/main",
    "origin/Calibration_analysis"
]

for branch in branches:

    result = subprocess.run(
        [
            "git",
            "-C", str(REPO_ROOT),
            "ls-tree",
            "-r",
            "--name-only",
            branch
        ],
        capture_output=True,
        text=True,
        check=True
    )

    tracked_files = result.stdout.splitlines()

    print(f"\n===== {branch} =====")

    for filename in required_names:

        matches = [
            path for path in tracked_files
            if Path(path).name == filename
        ]

        if matches:
            print("FOUND:", matches[0])
        else:
            print("MISSING:", filename)

# Check if Git is ignoring the metadata directory
print("\n===== Git ignore check =====")

result = subprocess.run(
    [
        "git",
        "-C", str(REPO_ROOT),
        "check-ignore",
        "-v",
        "--no-index",
        "metadata/breakhis_patient_split.csv"
    ],
    capture_output=True,
    text=True
)

print(
    result.stdout.strip()
    or "No matching ignore rule found."
)


===== origin/main =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== origin/Calibration_analysis =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== Git ignore check =====
No matching ignore rule found.


In [ ]:

# ============================================================
# Load and verify frozen Development metadata
# ============================================================

import pandas as pd

image_metadata = pd.read_csv(
    required_files["final_40x_metadata"],
    dtype={"patient_id": str}
)

patient_split = pd.read_csv(
    required_files["patient_split"],
    dtype={"patient_id": str}
)

development_folds = pd.read_csv(
    required_files["development_folds"],
    dtype={"patient_id": str}
)

train_patches = pd.read_csv(
    required_files["training_patch_manifest"],
    dtype={"patient_id": str}
)

val_patches = pd.read_csv(
    required_files["validation_patch_manifest"],
    dtype={"patient_id": str}
)

# Standardize labels without changing stored files
for df in [image_metadata, patient_split, development_folds]:
    df["split"] = df["split"].str.strip().str.lower()

development_folds["development_fold"] = (
    development_folds["development_fold"].astype(int)
)

# Verify original data dimensions
assert len(image_metadata) == 1961
assert len(patient_split) == 81
assert len(development_folds) == 49

assert len(train_patches) == 12034
assert len(val_patches) == 5861

assert image_metadata["patient_id"].nunique() == 81
assert patient_split["patient_id"].nunique() == 81
assert development_folds["patient_id"].nunique() == 49

assert not image_metadata["filename"].duplicated().any()
assert not patient_split["patient_id"].duplicated().any()
assert not development_folds["patient_id"].duplicated().any()

assert image_metadata["magnification"].astype(str).eq("40").all()

# Frozen split membership
DEV_IDS = set(
    patient_split.loc[
        patient_split["split"] == "development",
        "patient_id"
    ]
)

CAL_IDS = set(
    patient_split.loc[
        patient_split["split"] == "calibration",
        "patient_id"
    ]
)

TEST_IDS = set(
    patient_split.loc[
        patient_split["split"] == "test",
        "patient_id"
    ]
)

assert len(DEV_IDS) == 49
assert len(CAL_IDS) == 16
assert len(TEST_IDS) == 16

assert DEV_IDS.isdisjoint(CAL_IDS)
assert DEV_IDS.isdisjoint(TEST_IDS)
assert CAL_IDS.isdisjoint(TEST_IDS)

assert (
    DEV_IDS | CAL_IDS | TEST_IDS
) == set(patient_split["patient_id"])

# Check image split agrees with authoritative patient split
split_lookup = patient_split.set_index("patient_id")["split"]

assert (
    image_metadata["split"]
    == image_metadata["patient_id"].map(split_lookup)
).all()

# Development subset
dev_images = image_metadata.loc[
    image_metadata["patient_id"].isin(DEV_IDS)
].copy()

assert len(dev_images) == 1133
assert dev_images["patient_id"].nunique() == 49

# Verify original Development folds
assert set(development_folds["patient_id"]) == DEV_IDS
assert development_folds["split"].eq("development").all()

fold_counts = (
    development_folds["development_fold"]
    .value_counts()
    .sort_index()
)

assert fold_counts.to_dict() == {
    1: 17,
    2: 16,
    3: 16
}

print("Development patients:", len(DEV_IDS))
print("Calibration patients:", len(CAL_IDS))
print("Test patients:", len(TEST_IDS))

print("\nDevelopment images:", len(dev_images))

print("\nDevelopment fold counts:")
display(fold_counts)

print("\nPASS: Frozen Development metadata verified.")


Development patients: 49
Calibration patients: 16
Test patients: 16

Development images: 1133

Development fold counts:


,count
development_fold,
1,17
2,16
3,16



PASS: Frozen Development metadata verified.


In [ ]:

# ============================================================
# Reconstruct Development patch manifest
# ============================================================

all_patches = pd.concat(
    [train_patches, val_patches],
    ignore_index=True
)

# Check total number of patches and membership
assert len(all_patches) == 17895

assert all_patches["patient_id"].nunique() == 49
assert all_patches["filename"].nunique() == 1133

assert set(all_patches["patient_id"]) == DEV_IDS
assert set(all_patches["filename"]) == set(dev_images["filename"])

# Check for repeated patch identities
assert not all_patches.duplicated(
    ["filename", "patch_index"]
).any()

# Build authoritative image lookup
image_lookup = dev_images.set_index("filename")

for column in [
    "patient_id",
    "diagnosis",
    "relative_path"
]:

    expected = all_patches["filename"].map(
        image_lookup[column]
    )

    assert expected.notna().all(), (
        f"Missing expected values for {column}"
    )

    assert (
        all_patches[column].astype(str).to_numpy()
        == expected.astype(str).to_numpy()
    ).all(), (
        f"Patch/image mismatch in {column}"
    )

# Verify binary labels
expected_labels = (
    all_patches["filename"]
    .map(image_lookup["diagnosis"])
    .eq("Malignant")
    .astype(int)
)

assert (
    all_patches["label"].astype(int).to_numpy()
    == expected_labels.to_numpy()
).all()

# Verify all patches use valid indices
assert all_patches["patch_index"].between(0, 15).all()
assert all_patches["tissue_fraction"].between(0, 1).all()

patches_per_image = (
    all_patches.groupby("filename").size()
)

assert patches_per_image.between(1, 16).all()

# Confirm that the old preliminary split used Fold 1
fold_lookup = development_folds.set_index(
    "patient_id"
)["development_fold"]

original_val_folds = set(
    val_patches["patient_id"].map(fold_lookup)
)

original_train_folds = set(
    train_patches["patient_id"].map(fold_lookup)
)

assert original_val_folds == {1}
assert original_train_folds == {2, 3}

assert set(train_patches["patient_id"]).isdisjoint(
    set(val_patches["patient_id"])
)

# Attach original frozen Development fold
all_patches["development_fold"] = (
    all_patches["patient_id"].map(fold_lookup)
)

assert all_patches["development_fold"].notna().all()

all_patches["development_fold"] = (
    all_patches["development_fold"].astype(int)
)

all_patches = all_patches.sort_values(
    [
        "development_fold",
        "patient_id",
        "filename",
        "patch_index"
    ]
).reset_index(drop=True)

print("Development patients:", all_patches["patient_id"].nunique())
print("Development images:", all_patches["filename"].nunique())
print("Development patches:", len(all_patches))

print("\nPatches per image:")
print(patches_per_image.describe())

print("\nPASS: Frozen Development patch manifest reconstructed.")


Development patients: 49
Development images: 1133
Development patches: 17895

Patches per image:
count    1133.000000
mean       15.794351
std         1.121223
min         1.000000
25%        16.000000
50%        16.000000
75%        16.000000
max        16.000000
dtype: float64

PASS: Frozen Development patch manifest reconstructed.


In [ ]:

# ============================================================
# Prepare patient-grouped three-fold CV
# ============================================================

cv_splits = {}
cv_summary_rows = []

for validation_fold in [1, 2, 3]:

    train_df = all_patches.loc[
        all_patches["development_fold"] != validation_fold
    ].copy()

    val_df = all_patches.loc[
        all_patches["development_fold"] == validation_fold
    ].copy()

    train_patients = set(train_df["patient_id"])
    val_patients = set(val_df["patient_id"])

    # No shared patients
    assert train_patients.isdisjoint(val_patients)

    # Together they cover all Development patients
    assert train_patients | val_patients == DEV_IDS

    # No held-out patients
    assert (
        train_patients | val_patients
    ).isdisjoint(CAL_IDS | TEST_IDS)

    # Save each fold in memory for later model training
    cv_splits[validation_fold] = {
        "train": train_df,
        "validation": val_df
    }

    cv_summary_rows.append({
        "validation_fold": validation_fold,
        "training_patients": len(train_patients),
        "validation_patients": len(val_patients),
        "training_images": train_df["filename"].nunique(),
        "validation_images": val_df["filename"].nunique(),
        "training_patches": len(train_df),
        "validation_patches": len(val_df)
    })

cv_summary = pd.DataFrame(cv_summary_rows)

display(cv_summary)

assert cv_summary["training_patients"].tolist() == [
    32, 33, 33
]

assert cv_summary["validation_patients"].tolist() == [
    17, 16, 16
]

assert cv_summary["validation_images"].sum() == 1133

assert cv_summary["validation_patches"].sum() == 17895

print(
    "\nPASS: All three patient-grouped Development "
    "folds are ready."
)


,validation_fold,training_patients,validation_patients,training_images,validation_images,training_patches,validation_patches
0,1,32,17,761,372,12034,5861
1,2,33,16,702,431,11085,6810
2,3,33,16,803,330,12671,5224



PASS: All three patient-grouped Development folds are ready.


In [ ]:

# ============================================================
# Save verified Development outputs
# ============================================================

from pathlib import Path

METADATA_DIR = REPO_ROOT / "Metadata"

RESULTS_DIR = (
    REPO_ROOT / "results" / "formal_development"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = (
    METADATA_DIR /
    "breakhis_development_all_patch_manifest.csv"
)

summary_path = (
    RESULTS_DIR /
    "development_cv_preflight_summary.csv"
)

all_patches.to_csv(manifest_path, index=False)
cv_summary.to_csv(summary_path, index=False)

# Reopen files to verify they were written correctly
saved_manifest = pd.read_csv(
    manifest_path,
    dtype={"patient_id": str}
)

saved_summary = pd.read_csv(summary_path)

assert len(saved_manifest) == 17895
assert saved_manifest["patient_id"].nunique() == 49

assert len(saved_summary) == 3

print("Saved files:")
print(manifest_path)
print(summary_path)

print("\nPASS: Stage A completed successfully.")


Saved files:
/content/breast-histology-calibration/Metadata/breakhis_development_all_patch_manifest.csv
/content/breast-histology-calibration/results/formal_development/development_cv_preflight_summary.csv

PASS: Stage A completed successfully.


# Stage B — Fine-Tuned ResNet-18

## GPU and Dataset Verification

This stage implements formal ResNet-18 model development using only the frozen 49-patient BreakHis Development cohort.

All three original Development folds will be used for model selection. Calibration, Test, BACH, and BRACS remain excluded.

In [ ]:

# ============================================================
# Verify GPU and PyTorch environment
# ============================================================

import torch
import torchvision
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(f"GPU memory: {gpu_memory:.2f} GB")

else:
    device = torch.device("cpu")

    raise RuntimeError(
        "GPU is not available. "
        "Select T4 GPU in Colab before continuing."
    )

print("\nPASS: GPU environment is ready.")


PyTorch version: 2.11.0+cu130
Torchvision version: 0.26.0+cu130

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

PASS: GPU environment is ready.


In [ ]:

# ============================================================
# Load frozen Development manifest
# ============================================================

from pathlib import Path
import subprocess
import pandas as pd

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

REPO_URL = (
    "https://github.com/Yisihaq01/"
    "breast-histology-calibration.git"
)

# Restore repository if the GPU runtime reset /content
if not REPO_ROOT.exists():

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_ROOT)
        ],
        check=True
    )

METADATA_DIR = REPO_ROOT / "Metadata"

RESULTS_DIR = (
    REPO_ROOT /
    "results" /
    "formal_development"
)

manifest_path = (
    METADATA_DIR /
    "breakhis_development_all_patch_manifest.csv"
)

assert manifest_path.exists(), (
    "Frozen Development manifest is missing. "
    "Restore the committed Stage A file before continuing."
)

all_patches = pd.read_csv(
    manifest_path,
    dtype={"patient_id": str}
)

assert len(all_patches) == 17895

assert (
    all_patches["patient_id"].nunique() == 49
)

assert (
    all_patches["filename"].nunique() == 1133
)

assert set(
    all_patches["development_fold"].unique()
) == {1, 2, 3}

print("Repository:", REPO_ROOT)

print("\nDevelopment patients:",
      all_patches["patient_id"].nunique())

print("Development images:",
      all_patches["filename"].nunique())

print("Development patches:", len(all_patches))

print("\nPatients by fold:")

display(
    all_patches.groupby("development_fold")[
        "patient_id"
    ].nunique()
)

print(
    "\nPASS: Frozen Development manifest "
    "loaded successfully."
)


Repository: /content/breast-histology-calibration

Development patients: 49
Development images: 1133
Development patches: 17895

Patients by fold:


,patient_id
development_fold,
1,17
2,16
3,16



PASS: Frozen Development manifest loaded successfully.


In [ ]:

# ============================================================
# Locate original BreakHis dataset
# ============================================================

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("kagglehub") is None:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "kagglehub"
        ],
        check=True
    )

import kagglehub

# Reuse an existing dataset location when available
known_root = Path("/kaggle/input/breakhis")

if known_root.exists():

    DATASET_ROOT = known_root

else:

    DATASET_ROOT = Path(
        kagglehub.dataset_download(
            "ambarish/breakhis"
        )
    )

print("BreakHis dataset root:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())

assert DATASET_ROOT.exists()

print("\nTop-level contents:")

for path in sorted(DATASET_ROOT.iterdir())[:15]:
    print("-", path.name)

print("\nPASS: BreakHis dataset located.")


Using Colab cache for faster access to the 'breakhis' dataset.
BreakHis dataset root: /kaggle/input/breakhis
Dataset exists: True

Top-level contents:
- BreaKHis_v1
- Folds.csv

PASS: BreakHis dataset located.


In [ ]:

# ============================================================
# Verify original image paths
# ============================================================

sample_rows = (
    all_patches[
        ["filename", "relative_path", "patient_id"]
    ]
    .drop_duplicates("filename")
    .sample(
        n=5,
        random_state=631
    )
)

print("Checking 5 frozen Development images:\n")

missing_images = []

for _, row in sample_rows.iterrows():

    image_path = (
        DATASET_ROOT /
        row["relative_path"]
    )

    if not image_path.exists():

        missing_images.append(str(image_path))

        print(
            "MISSING:",
            row["filename"]
        )

        continue

    with Image.open(image_path) as img:

        print(
            "FOUND:",
            row["filename"],
            "| Size:",
            img.size,
            "| Mode:",
            img.mode
        )

if missing_images:

    print("\nExample missing path:")
    print(missing_images[0])

    raise FileNotFoundError(
        "One or more frozen image paths could not be resolved. "
        "Check DATASET_ROOT before training."
    )

print(
    "\nPASS: Frozen Development image paths "
    "resolve correctly."
)


Checking 5 frozen Development images:

FOUND: SOB_M_DC-14-12312-40-007.png | Size: (700, 460) | Mode: RGB
FOUND: SOB_M_PC-14-15704-40-008.png | Size: (700, 460) | Mode: RGB
FOUND: SOB_M_MC-14-10147-40-004.png | Size: (700, 460) | Mode: RGB
FOUND: SOB_M_DC-14-13412-40-016.png | Size: (700, 460) | Mode: RGB
FOUND: SOB_M_DC-14-10926-40-004.png | Size: (700, 460) | Mode: RGB

PASS: Frozen Development image paths resolve correctly.


In [ ]:

# ============================================================
# Inspect original frozen preprocessing code
# ============================================================

import json
from pathlib import Path

# Find Notebook 01 in the cloned repository
notebook_candidates = [
    p for p in REPO_ROOT.rglob("*.ipynb")
    if "01_breakhis" in p.name.lower()
]

print("Matching notebooks:")

for path in notebook_candidates:
    print(path.relative_to(REPO_ROOT))

assert notebook_candidates, (
    "Notebook 01 was not found in the repository. "
    "Do not recreate preprocessing from memory."
)

original_notebook = notebook_candidates[0]

with open(
    original_notebook,
    "r",
    encoding="utf-8"
) as f:
    notebook_data = json.load(f)

keywords = [
    "0.49",
    "0.50",
    "lanczos",
    "resample",
    "physical_resolution",
    "target_mpp",
    "source_mpp",
    "resize_image",
    "patch_coordinates"
]

matching_cells = []

for index, cell in enumerate(
    notebook_data["cells"]
):

    if cell["cell_type"] != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    source_lower = source.lower()

    if any(
        keyword in source_lower
        for keyword in keywords
    ):
        matching_cells.append(
            (index, source)
        )

print(
    "\nMatching preprocessing cells:",
    len(matching_cells)
)

for index, source in matching_cells[:8]:

    print("\n" + "=" * 65)
    print("NOTEBOOK 01 — CELL", index)
    print("=" * 65)

    print(source[:5000])


Matching notebooks:
Notebooks/01_breakhis_data_audit_and_split.ipynb
Notebooks/01_breakhis_audit.ipynb

Matching preprocessing cells: 8

NOTEBOOK 01 — CELL 57
print("Development images evaluated:",
      len(development_tissue))

print("\nTissue-fraction summary:")
print(
    development_tissue["tissue_fraction"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

NOTEBOOK 01 — CELL 63
# ============================================================
# Frozen preprocessing parameters
# ============================================================

SOURCE_MPP_BREAKHIS = 0.49
TARGET_MPP = 0.50

PATCH_SIZE = 224
GRID_SIZE = 4
MAX_PATCHES = 16

MIN_PATCH_TISSUE_FRACTION = 0.20

OD_THRESHOLD = 0.08
SATURATION_THRESHOLD = 0.05

NOTEBOOK 01 — CELL 64
# ============================================================
# Physical-scale norma

In [ ]:

# ============================================================
# Audit all source paths and patch coordinates
# ============================================================

from PIL import Image
from collections import Counter

# One record per source image
unique_images = (
    all_patches[
        ["filename", "relative_path", "patient_id"]
    ]
    .drop_duplicates("filename")
    .copy()
)

assert len(unique_images) == 1133

missing_paths = []
image_sizes = Counter()

for row in unique_images.itertuples(index=False):

    image_path = (
        DATASET_ROOT / row.relative_path
    )

    if not image_path.is_file():
        missing_paths.append(str(image_path))
        continue

    with Image.open(image_path) as img:
        image_sizes[img.size] += 1

print("Development images checked:", len(unique_images))
print("Missing image paths:", len(missing_paths))

print("\nOriginal image dimensions:")

for size, count in sorted(image_sizes.items()):
    print(f"{size}: {count} images")

assert not missing_paths, (
    f"Missing source images: {missing_paths[:5]}"
)

# Verify saved patch coordinate integrity
assert all_patches[
    ["x", "y", "patch_index"]
].notna().all().all()

for column in ["x", "y", "patch_index"]:

    values = all_patches[column]

    assert (values >= 0).all()

    assert (
        values == values.astype(int)
    ).all()

assert all_patches["patch_index"].between(0, 15).all()

assert not all_patches.duplicated(
    ["filename", "patch_index"]
).any()

print("\nFrozen patch-coordinate ranges:")

print(
    "X:",
    all_patches["x"].min(),
    "to",
    all_patches["x"].max()
)

print(
    "Y:",
    all_patches["y"].min(),
    "to",
    all_patches["y"].max()
)

print(
    "Patch index:",
    all_patches["patch_index"].min(),
    "to",
    all_patches["patch_index"].max()
)

fallback_count = (
    all_patches["used_fallback"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

print(
    "\nFallback patches:",
    fallback_count
)

print(
    "\nPASS: All Development image paths "
    "and saved patch coordinates verified."
)


Development images checked: 1133
Missing image paths: 0

Original image dimensions:
(700, 456): 21 images
(700, 460): 1112 images

Frozen patch-coordinate ranges:
X: 0 to 462
Y: 0 to 227
Patch index: 0 to 15

Fallback patches: 0

PASS: All Development image paths and saved patch coordinates verified.


In [ ]:

# ============================================================
# Frozen preprocessing and transformations
# Reproduced from Notebook 01
# ============================================================

import random

import torch
from PIL import Image

from torchvision import transforms
from torchvision.transforms import functional as TF


# Frozen preprocessing parameters
SOURCE_MPP_BREAKHIS = 0.49
TARGET_MPP = 0.50

PATCH_SIZE = 224


def resample_to_target_mpp(
    image,
    source_mpp=SOURCE_MPP_BREAKHIS,
    target_mpp=TARGET_MPP
):
    """
    Resample the source image to 0.50 micrometers/pixel.
    Preserve the original Notebook 01 implementation.
    """

    image = image.convert("RGB")

    scale = source_mpp / target_mpp

    new_width = round(image.width * scale)
    new_height = round(image.height * scale)

    return image.resize(
        (new_width, new_height),
        Image.Resampling.LANCZOS
    )


class RandomRightAngleRotation:

    def __call__(self, image):

        angle = random.choice(
            [0, 90, 180, 270]
        )

        return TF.rotate(
            image,
            angle
        )


# Training: geometric augmentation only
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    RandomRightAngleRotation(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Validation: no random augmentation
val_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Verify original physical-resolution behavior
example_size = (700, 460)

example_image = Image.new(
    "RGB",
    example_size
)

resampled_example = resample_to_target_mpp(
    example_image
)

assert resampled_example.size == (686, 451)

print("Original image:", example_size)
print("Resampled image:", resampled_example.size)

print("\nTraining transformations:")
print(train_transform)

print("\nValidation transformations:")
print(val_transform)

print(
    "\nPASS: Frozen preprocessing "
    "and transformations restored."
)


Original image: (700, 460)
Resampled image: (686, 451)

Training transformations:
Compose(
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Validation transformations:
Compose(
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

PASS: Frozen preprocessing and transformations restored.


In [ ]:

# ============================================================
# BreakHis patch Dataset
# ============================================================

from pathlib import Path

from torch.utils.data import Dataset, DataLoader


class BreakHisPatchDataset(Dataset):

    def __init__(
        self,
        manifest,
        dataset_root,
        transform=None
    ):

        self.manifest = (
            manifest
            .reset_index(drop=True)
            .copy()
        )

        self.dataset_root = Path(
            dataset_root
        )

        self.transform = transform

        required_columns = {
            "relative_path",
            "patient_id",
            "filename",
            "label",
            "patch_index",
            "x",
            "y"
        }

        missing_columns = (
            required_columns
            - set(self.manifest.columns)
        )

        if missing_columns:
            raise ValueError(
                f"Missing manifest columns: "
                f"{missing_columns}"
            )

        if not set(
            self.manifest["label"].unique()
        ).issubset({0, 1}):
            raise ValueError(
                "Dataset labels must be binary."
            )

    def __len__(self):

        return len(self.manifest)

    def __getitem__(self, index):

        row = self.manifest.iloc[index]

        image_path = (
            self.dataset_root /
            row["relative_path"]
        )

        # Load original-resolution source image
        with Image.open(image_path) as source:
            image = resample_to_target_mpp(
                source
            )

        # Use frozen patch coordinates
        x = int(row["x"])
        y = int(row["y"])

        # Prevent accidental padded/out-of-bounds crops
        if (
            x < 0
            or y < 0
            or x + PATCH_SIZE > image.width
            or y + PATCH_SIZE > image.height
        ):
            raise ValueError(
                "Invalid frozen crop coordinates: "
                f"{row['filename']}, x={x}, y={y}, "
                f"resampled_size={image.size}"
            )

        patch = image.crop(
            (
                x,
                y,
                x + PATCH_SIZE,
                y + PATCH_SIZE
            )
        )

        if self.transform is not None:
            patch = self.transform(patch)

        label = torch.tensor(
            float(row["label"]),
            dtype=torch.float32
        )

        return {
            "image": patch,
            "label": label,
            "patient_id": str(row["patient_id"]),
            "filename": str(row["filename"]),
            "patch_index": int(row["patch_index"])
        }


print("PASS: BreakHisPatchDataset defined.")


PASS: BreakHisPatchDataset defined.


In [ ]:

# ============================================================
# Prepare Fold 1 DataLoaders
# ============================================================

import numpy as np
import random
import torch

from torch.utils.data import DataLoader


VALIDATION_FOLD = 1
BATCH_SIZE = 32
NUM_WORKERS = 2

# Frozen patient-grouped assignments
fold_train_df = (
    all_patches[
        all_patches["development_fold"]
        != VALIDATION_FOLD
    ]
    .copy()
)

fold_val_df = (
    all_patches[
        all_patches["development_fold"]
        == VALIDATION_FOLD
    ]
    .copy()
)

# Patient-level leakage checks
train_patient_ids = set(
    fold_train_df["patient_id"]
)

val_patient_ids = set(
    fold_val_df["patient_id"]
)

assert train_patient_ids.isdisjoint(
    val_patient_ids
)

assert len(train_patient_ids) == 32
assert len(val_patient_ids) == 17

assert len(fold_train_df) == 12034
assert len(fold_val_df) == 5861


# Construct datasets
train_dataset = BreakHisPatchDataset(
    manifest=fold_train_df,
    dataset_root=DATASET_ROOT,
    transform=train_transform
)

val_dataset = BreakHisPatchDataset(
    manifest=fold_val_df,
    dataset_root=DATASET_ROOT,
    transform=val_transform
)


# Seed independent DataLoader worker processes
def seed_worker(worker_id):

    worker_seed = (
        torch.initial_seed() % (2 ** 32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(631)


# Training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)


# Validation DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


print("Fold:", VALIDATION_FOLD)

print(
    "Training patients:",
    len(train_patient_ids)
)

print(
    "Validation patients:",
    len(val_patient_ids)
)

print(
    "\nTraining patches:",
    len(train_dataset)
)

print(
    "Validation patches:",
    len(val_dataset)
)

print(
    "\nTraining batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "\nPASS: Fold 1 DataLoaders constructed."
)


Fold: 1
Training patients: 32
Validation patients: 17

Training patches: 12034
Validation patches: 5861

Training batches: 377
Validation batches: 184

PASS: Fold 1 DataLoaders constructed.


In [ ]:

# ============================================================
# Test training and validation batches
# ============================================================

train_batch = next(
    iter(train_loader)
)

val_batch = next(
    iter(val_loader)
)


def verify_batch(batch, name):

    images = batch["image"]
    labels = batch["label"]

    print(f"\n{name} batch:")

    print("Image tensor:", images.shape)
    print("Label tensor:", labels.shape)

    print(
        "Distinct patients:",
        len(set(batch["patient_id"]))
    )

    print(
        "Distinct source images:",
        len(set(batch["filename"]))
    )

    print(
        "Labels:",
        torch.unique(labels).tolist()
    )

    # N x 3 x 224 x 224
    assert images.ndim == 4
    assert images.shape[1:] == (3, 224, 224)

    assert labels.ndim == 1
    assert labels.shape[0] == images.shape[0]

    assert torch.isfinite(images).all()
    assert torch.isfinite(labels).all()

    assert set(
        torch.unique(labels).tolist()
    ).issubset({0.0, 1.0})

    assert len(batch["patient_id"]) == len(labels)
    assert len(batch["filename"]) == len(labels)

    print(f"PASS: {name} batch verified.")


verify_batch(
    train_batch,
    "Training"
)

verify_batch(
    val_batch,
    "Validation"
)


# Transfer one batch to GPU
gpu_images = train_batch["image"].to(
    device,
    non_blocking=True
)

gpu_labels = train_batch["label"].to(
    device,
    non_blocking=True
)

assert gpu_images.is_cuda
assert gpu_labels.is_cuda

print("\nGPU tensor shape:", gpu_images.shape)

print(
    "GPU device:",
    gpu_images.device
)

print(
    "\nPASS: Training and validation "
    "DataLoaders are ready for ResNet-18."
)



Training batch:
Image tensor: torch.Size([32, 3, 224, 224])
Label tensor: torch.Size([32])
Distinct patients: 20
Distinct source images: 32
Labels: [0.0, 1.0]
PASS: Training batch verified.

Validation batch:
Image tensor: torch.Size([32, 3, 224, 224])
Label tensor: torch.Size([32])
Distinct patients: 1
Distinct source images: 2
Labels: [0.0]
PASS: Validation batch verified.

GPU tensor shape: torch.Size([32, 3, 224, 224])
GPU device: cuda:0

PASS: Training and validation DataLoaders are ready for ResNet-18.
